# Acervo recuperado — versão sanitizada

Material de estudo de 2025, com trechos fornecidos pela disciplina e respostas na cópia de Luigi. Créditos preservados quando explícitos. Saídas, imagens incorporadas, contatos e links privados foram removidos. Não é uma versão corrigida nem há garantia de execução. Veja `../docs/PROCEDENCIA.md` e os notebooks revisados em `../notebooks/`.


# Aritmética de Ponto Flutuante e Eliminação de Gauss

### Prof. Pedro Peixoto <[contato removido]>

Jan 2025

---




**Data de entrega:** Verifique no Classroom!

**Como entregar:** Enviar o seu jupyter notebook (.ipynb) pelo próprio Classroom. Uma cópia deste será automaticamente criada para você no seu Google Drive, você pode usar essa cópia mesmo, sem necessidade de criar outra.


**Leia com atenção:**


*   Você deve fazer obrigatoriamente todas as atividades marcadas com a exclamação❗.
*   Atividades marcadas com 🧩 são opcionais, mas interessantes, podendo ser desafiadoras.
*   Faça uma cópia do notebook para o seu Google Drive e use o Google Colab para executar o notebook e fazer a tarefa.
*   A atividade vai ser baseada no seu número USP para que as soluções dos alunos fiquem diferentes.
*   Há questões dissertativas!
*   Siga o passo a passo indicado no tutorial, respondendo às atividades propostas no seu jupyter notebook.
*   O notebook que entregar deve estar 100% funcional, isto é, todos os códigos rodando no Colab sem erros.
*   Você deverá entregar apenas uma cópia do jupyter notebook (.ipynb) no Classroom, na funcionalidade de "entregar" do Classroom.




###❗Atividade 0❗
Para mostrar que você leu até aqui, escreva seu nome completo (assinatura) na linha a seguir:

*Assinatura:* Luigi Scofano de Araujo



---





Primeiro vamos carregar algumas bibliotecas: Numpy, Scipy Linear Algebra (scipy.linalg), System (sys) e Time (time)

In [ ]:
import numpy as np
import scipy.linalg as la
import sys
import time

Agora vamos definir alguns números especiais ("magic") para cada aluno, com base no seu número USP

In [ ]:
# Coloque aqui o seu número USP
nusp = 13680334

Estes números serão importantes para a atividade:

In [ ]:
magic_ilong = int(str(nusp).replace("0",""))
magic_ishort = int(str(magic_ilong)[:2])
print("magic_ilong  =", magic_ilong)
print("magic_ishort =", magic_ishort)



---




## Aritmética de Ponto Flutuante

Começamos esse tutorial discutindo um pouco de aritmética de ponto flutuante, que vamos definir mais adiante.



Algumas referências:

[1]https://en.wikipedia.org/wiki/IEEE_754

[2] https://betterprogramming.pub/floating-point-numbers-are-weird-in-python-heres-how-to-fix-them-51336e4ad51a

[3] https://ironpython-test.readthedocs.io/en/latest/tutorial/floatingpoint.html


A matemática está furada no computador?

In [ ]:
a = 10
b = 0.1
c = 0.2

In [ ]:
r1 = a*(b + c)
r2 = a*b + a*c

r1 e r2 deveriam ser iguais...Mas...

In [ ]:
print(r1 == r2)

In [ ]:
print(r1, r2, r1-r2)

A distributiva e associatividade não valem no computador ?!?!

In [ ]:
.2 + .2 + .2 == 0.6

In [ ]:
sum = 0.0
n = 10
for i in range(n):
    sum += 1/n
print(sum)

Vamos tentar entender os motivos partindo bem do princípio: como o computador representa os números!

### Números inteiros no computador

Suponha que tenho apenas 3 bits para representar inteiros (ignore o sinal). Ou seja, temos os números binários
000, 001, 011, 100, 110, ..., 111
que representam respectivamente
0, 1, 2, 3, 4, 5, ..., 7
sendo 7 o maior número que conseguimos representar.

Vejamos isso em Python:

In [ ]:
a = 5
b = 6

In [ ]:
bin_a=bin(a)
bin_b=bin(b)
print(a, bin_a)
print(b, bin_b)

Os dois números estão bem representados, mas...

In [ ]:
print(a+b, bin(a+b))

isso não é representável em 3 bits (o Python, espertinho, percebeu isso e então colocou mais um bit para representar o número 11...).

Em um computador moderno teríamos 64bits para representar um inteiro, e um sinal para carregar.

Veja aqui mais detalhes: https://prof-thiagooliveira.netlify.app/post/signed-and-unsigned-binary-numbers/

[Imagem histórica omitida; consulte a referência textual adjacente.]



---



>E no Python, quantos bits ele usa?

Por "sorte", o Python usa _long int_ e estende a precisão no caso de inteiros se necessário, então é difícil termos problemas com inteiros em Python...Ufa!



---




### Números "reais" no computador

Agora começa o problema: como o computador representa números reais no computador?

Ele usa o que é conhecido como *ponto flutuante*, que nada mais é que uma padronização da nossa notação científica:




[Imagem histórica omitida; consulte a referência textual adjacente.]

Mas **cuidado**, pois o computador não trabalha com base 10, e sim base 2 (binária), com ponto flutuante. Para facilitar o entendimento de alguns processos, muitos textos adotam a base 10, mas sempre temos que ter em mente que no fundo será usado a base 2.

Em 32 bits, no padrão IEEE-754 temos:


[Imagem histórica omitida; consulte a referência textual adjacente.]

Isto é:


*   1 bit para o sinal (+/-)
*   8 bits para o expoente. Como precisamos de um sinal para o expoente, por convenção o expoente é subtraído por 127, que é 01111111 em binário (em precisão simples, float).
*   23 bits para a fração (mantissa)



*Desafio:* a parte da "fração", que gostaríamos que representasse casas "decimais", precisa ser montada com base binária! Ou seja, muitos números não serão passíveis de serem representados, ou são representados com algum erro.

Vejam alguns exemplos :





---


**Número "1"** em ponto flutuante é dado, em 32 bits, por:

$\color{blue}{0}\color{green}{01111111}\color{red}{00000000000000000000000}$

onde as cores indicam o $\color{blue}{sinal}$, o $\color{green}{expoente}$,  e a $\color{red}{mantissa}$, respectivamente.

Traduzindo (em binário ainda...):

$ (−1)^\color{blue}{0}×​2^{(\color{green}{01111111} − 01111111)​}\times ​1.\color{red}{00000000000000000000000} $

Traduzindo para decimal:
$(−1)^\color{blue}{0}×​2^{\color{green}{0}}×\color{red}{​1.0} = 1.0 $ (exato!).



---



**Número "0.5"** é só um pouco diferente do 1 em ponto flutuante é dado, em 32 bits:

No computador:
$\color{blue}{0}\color{green}{01111110}\color{red}{00000000000000000000000}$

Binário:
$ (−1)^\color{blue}{0}×​2^{(\color{green}{01111110} − 01111111)​}\times ​1.\color{red}{00000000000000000000000} $

Decimal:
$(−1)^\color{blue}{0}×​2^{\color{green}{-1}}×\color{red}{​1.0} = 0.5$ (exato!).



---


**Número "0.05"** complica, e muito, vejamos:

No computador: $\color{blue}{0}\color{green}{01111010}\color{red}{10011001100110011001101}$

Binário: $ (−1)^\color{blue}{0}×​2^{(\color{green}{01111010} − 01111111)​}\times ​1.\color{red}{10011001100110011001101} $

Decimal:
$(−1)^\color{blue}{0}×​2^{\color{green}{-5}}×1.\color{red}{60000002384185791015625} \approx 0.0500000007450580596923828125​$ - Não conseguimos representar exatamente :-( .


---



Referência: https://float.exposed/0xb3bc8b0f



---


### ❗**Atividade 1**❗


1.   Qual o maior número representável em Python por aritmética de ponto flutuante no sistema que você está usando? (DICA: use a biblioteca sys.float_info (https://docs.python.org/3/library/sys.html) para saber mais sobre como o seu sistema trabalha com o tipo "float")

2.   O que acontece se chamar em Python um número maior que esse? Ou se fizer uma conta que resulte em um número maior que esse? (Dê um exemplo)

3. Quantos dígitos são bem representados nesse sistema de aritmética de ponto flutuante? (DICA: use o sys.float_info)

4. Qual o resultado da conta 1.0e-23 + 1.0e-23? Justifique.

5. Qual o resultado da conta 1.0 + 1.0e-23? Justifique.

6. Qual o resultado da conta 1.0e+14 + 1.0e-14? Justifique.

7. 🧩 (opcional) Leia o tutorial https://docs.python.org/pt-br/3/tutorial/floatingpoint.html e comente por que o computador não consegue representar o número 0.1 exatamente. Qual número aproximado é alocado na memória e usado nas contas?






In [ ]:
# Códigos usados na resposta ilustrando cada caso

# 1. Qual o maior número representável em Python por aritmética de ponto flutuante
maior_float = sys.float_info.max
print(f"1. Maior número representável: {maior_float}")

# 2. O que acontece se chamarmos um número maior que o max?
overflow_exemplo = maior_float * 2  # Deve resultar em 'inf'
print(f"2. Exemplo de overflow: {overflow_exemplo}")

# 3. Quantos dígitos são bem representados nesse sistema?
digitos_precisos = sys.float_info.dig
print(f"3. Dígitos bem representados: {digitos_precisos}")

# 4. Resultado de 1.0e-23 + 1.0e-23
soma1 = 1.0e-23 + 1.0e-23
print(f"4. 1.0e-23 + 1.0e-23 = {soma1}")

# 5. Resultado de 1.0 + 1.0e-23
soma2 = 1.0 + 1.0e-23  # Pequena soma que pode não ser representada corretamente
print(f"5. 1.0 + 1.0e-23 = {soma2}")

# 6. Resultado de 1.0e+14 + 1.0e-14
soma3 = 1.0e+14 + 1.0e-14  # Erro de precisão esperado
print(f"6. 1.0e+14 + 1.0e-14 = {soma3}")

# 7. Por que o computador não consegue representar 0.1 exatamente?
n_approx = format(0.1, ".55f")  # Representação interna de 0.1
print(f"7. Representação aproximada de 0.1: {n_approx}")


**Responder aqui com texto as justificativas e observações:**



1. O maior número representável em Python por aritmética de ponto flutuante segue o padrão IEEE-754 de precisão dupla. Ele pode ser encontrado com sys.float_info.max e representa o maior valor finito que pode ser armazenado sem se tornar infinito. Nesse caso: 1.7976931348623157e+308

2. Se multiplicarmos esse número max (1.7976931348623157e+308) por 2, ele ultrapassa o limite do IEEE-754 e o Python retorna 'inf' (infinito), caracterizando um **overflow** numérico.

3. O número de dígitos decimais que podem ser representados corretamente sem erro de arredondamento é 15 (obtido com a função sys.float_info.dig). Isso define quantos dígitos são confiáveis em cálculos de ponto flutuante.

4. A soma de 1.0e-23 + 1.0e-23 = 2e-23 funciona corretamente porque ambos os números possuem a mesma ordem de grandeza e estão dentro da precisão do formato de ponto flutuante, garantindo que não haja perda de significância.

  **OBS:** É importante pontuar que essa soma funciona, mesmo com 23 casas decimais, sendo que a precição máxima vista no item 3 é de 15 casas, pois a precisão de 15 dígitos se refere à quantidade de dígitos significativos em um número e não à posição decimal onde ele se encontra.

5. Já a soma de 1.0 + 1.0e-23 resulta em 1.0, pois 1.0e-23 é um valor extremamente pequeno comparado a 1.0. Como a precisão de um número de ponto flutuante é limitada em 15 casas significativas, a parte menor pode ser descartada.

6. No caso de 1.0e+14 + 1.0e-14, o erro de precisão ocorre porque 1.0e-14 é muito pequeno em relação a 1.0e+14, e pode acabar sendo ignorado na operação devido às limitações da precisão da mantissa de 15 digitos signiifcativos.

7. (OPCIONAL) O número 0.1 não pode ser representado exatamente em binário porque sua fração binária é infinita e periódica. Como o IEEE-754 utiliza 53 bits de mantissa, a precisão é limitada a cerca de 15 dígitos decimais significativos, o que causa erros de arredondamento. Por isso a representação de 0.1 possui muitas casas decimais.






---


### Simulando aritmética de ponto flutuante

Uma forma de simular a aritmética de ponto flutuante em diferentes tipos de computadores, com mais ou menos "bits" para representação numérica, é considerando o número de **algarismos significativos**.

Dada uma "máquina" com uma certa precisão, definimos o número de **algarismos significativos** como sendo o número de casas decimais, ou dígitos, que o computador consegue representar depois de transformado em notação científica.

Ex: Escreva os números na notação científica com 0,AAAAA x 10^B e verifique que

1000000 : possui 1 alg. sig.

0.00001 : possui 1 alg. sig.

1.00001 : possui 6 alg. sig.

0.12345 : possui 5 alg. sig.

Qualquer conta realizada deve ser ajustada (arredondada) no final para ficar nessa representação!



**Cuidado!**

Essa representação com algarismos significativos na base 10 é uma simplificação para fins didáticos, pois lembre-se que o computador trabalha com base 2.







---



---




Para saber esse número de algarismos (dígitos) significativos na sua máquina, use:

In [ ]:
print(sys.float_info.dig)

Em Python, podemos simular esses arredondamentos a cada operação usando a seguinte função:

In [ ]:
from math import floor, log10
# Uma opção para simular algarismos significativos
def round_alg_sig(x, sig_figs):
    # Recebe
    #    - x: numero do tipo float
    #    - sig_figs: número de algarismos significativos a ser usado
    # Devolve
    #    - Número arredondado para a aprecisão desejada

    if abs(x) > 0.0 and np.isfinite(x):
      x = float(x)
      precision = int(sig_figs)
      # Uso o round de casas decimais, deslocando as casas decimais para a posição correta
      return round(x, -int(floor(log10(abs(x)))) + (precision - 1))
    else:
      return x

#!pip install sigfig
#from sigfig import round



Um computador com 16bit tem 3 algarismos significativos (10 bits de mantissa)

(precisão conhecida como "half precision float")


Um exemplo tentando calcular $\tan(\pi/2-0.001)$ numericamente:

In [ ]:
# Computador de 16 bits (3 alg. sig.)
dig = 3

# Números em precisão dupla (64 bits)
# Python usa IEEE-754 double precision float
pi_double = np.pi
tan_double = np.tan(pi_double/2.0-0.001) #deveria ser bem grande!
print("Precisão dupla: ", pi_double, tan_double)

# Números em precisão "meia" (16 bits) simulada
# Todas as operações devem ser arredondadas!
pi_half = round_alg_sig(pi_double, dig)
tan_half = round_alg_sig(np.tan(round_alg_sig(round_alg_sig(pi_half/2.0,dig)-0.001,dig)), dig) #deveria ser bem grande!

print("Precisão meia :", pi_half, tan_half)




---



## Resolvendo Sistemas Lineares - Eliminação Gaussiana

### O problema

$$ Ax=b $$

Para os problemas a seguir, você deve usar uma matriz especial, construída conforme o código abaixo. Há 3 parâmetros importantes na geração da matriz:

* **n** : o tamanho da matriz
* **magic_ilong** : o seu número mágico, que fará sua matriz ser única
* **cond_pow** : Um número inteiro que quando maior, torna o sistema mais difícil de ser resolvido, por estar mais propenso a erros de arredondamento. Este número será usado na contrução do *número de condição* da matriz gerada (https://pt.wikipedia.org/wiki/N%C3%BAmero_de_condicionamento).

In [ ]:
# Sistema linear para testes - específica para cada aluno
# ------------------------------------------------------------
# Entrada:
#  n: tamanho da matriz
#  magic_ilong: seed de geração de números aleatórios
#  cond_pow : potência usada na definição do número de condição (inteiro positivo)
#
# Saída:
#   A: Matrix do sistema linear Ax=b
#   b: lado diretiro do sistema Ax=b
#   a: matriz aumentada [A|b]
def meu_sistema_linear(n=100, magic_ilong=magic_ilong, cond_pow=8):


  # Tamanho do sistema
  #n = 20

  np.random.seed(magic_ilong)

  # Gera matriz aleatória com número de condição alto usando decomposição QR
  cond = np.log(10**cond_pow) # Numero de condição
  exp_vec = np.arange(-cond/4., cond * (n + 1)/(4 * (n - 1)), cond/(2.*(n-1)))[:n]
  s = np.exp(exp_vec)
  S = np.diag(s)
  U, _ = la.qr((np.random.rand(n, n) - 5.) * 200)
  V, _ = la.qr((np.random.rand(n, n) - 5.) * 200)
  A = U.dot(S).dot(V.T)
  A = A.dot(A.T) #Simétrica

  #Vamos bagunçar um pouco a ordem da matriz
  trocas = np.random.randint(0, n, size=(int(n/2), 2))
  for i, j in trocas:
    A[[i, j], :] = A[[j, i], :]


  #Vetor b - vamos gerar b tal que a solução seja sempre um vetor com 1
  x = np.ones((n))
  b = A@x

  #Matrix aumentada a=[A|b]
  a = np.c_[A,b]

  np.set_printoptions(precision=3, suppress=True)
  #print("A:" , A[:10, :10]) #imprimir só o começo da matriz
  #print()
  #print("b:", b[:10]) #imprimir só o começo do vetor b
  #print()
  #print("a:", a) #imprimir a matriz aumentada

  return A, b, a

#### 🧩 *Reflexão*

Você consegue explicar como essa matriz é gerada e discutir suas propriedades a partir desse código?



---



---



O algoritmo clássico, conforme visto em aula:

In [ ]:
# Método Clássico de Eliminação Gaussiana
#
# O método recebe como entrada uma matrix aumentada a=[A|b],
# com dimensões n linhas por n+1 colunas, tendo b na última
#
# O método retorna a solução do sistema linear Ax=b,
#            ou um erro quando isso não for possível
def elim_gauss(a_orig):
  #Faço uma cópia, para não estragar a matriz original
  a = np.copy(a_orig)

  #Número de linhas e colunas
  n,m = a.shape

  # Vetor da solução
  x = np.empty((n))

  #Verifica se usuário forneceu uma matriz no formato certo
  if m != n+1 :
    print("Essa matriz não tem dimensões adequadas:", n, m)
    return x

  # Escalonamento
  zero_eps = 10e-16
  for j in range(n): # Colunas
      #print(j, a)
      #Caso o pivot seja zero (ou próximo de zero)
      if np.abs(a[j,j]) < zero_eps:
          #Pivotar linha i com linha linha com pivot não nulo!
          ktroca = j
          for k in range(j+1,n):
              if np.abs(a[k,j]) > zero_eps:
                  ktroca = k
                  break
          #Troca as linhas
          if (ktroca != j):
            print("Troquei linha ", j, " pela linha ", ktroca)
            a[[j, ktroca], :] = a[[ktroca, j], :]
          else:
            print("Esse sistema tem determinante nulo, ou muito perto de zero.")
            return x

      #Escalonamento
      for i in range(j+1, n): #linhas
          ratio = a[i,j]/a[j,j]
          for k in range(n+1):
              a[i,k] = a[i,k] - ratio * a[j,k]

  # Substituição
  x[n-1] = a[n-1,n]/a[n-1,n-1]

  for i in range(n-2,-1,-1):
      x[i] = a[i,n]

      for j in range(i+1,n):
          x[i] = x[i] - a[i,j]*x[j]

      x[i] = x[i]/a[i,i]

  return x

Vamos fazer um teste com uma matriz aleatória especial

Vamos testar o nosso esquema usando uma matriz com gerada com parâmetros:

* n = 300

In [ ]:
A, b, a = meu_sistema_linear(300, magic_ilong)

start_time = time.time()
x = elim_gauss(a)
tempo_orig = time.time() - start_time

print("Solução", x[:10]) #imprime só alguns elementos da solução
print("\n\nResíduo max|b-Ax|: ", np.max(np.abs(b-A@x)), " Tempo que levou: ", tempo_orig, " segundos")


**Desafios**


*   Esse método que implementamos parece funcionar, mas está muito lento! Como podemos melhorar a sua performance?

*   O resíduo parece bem grande, por que não está mais perto de 10e-16? (que seria o erro de arredondamento esperado para uma aritmética de ponto flutuante com 64 bits (double float)).

Vamos discutir esses dois pontos a seguir.



###Versão vetorizada do método de eliminação Gaussiana

Python é uma linguagem "interpretada", então cada linha é lida e executada individualmente, sem qualquer otimização para eficiência. Isso faz com que "loops" (for) em Python sejam bem lentos.

Podemos otimizar códigos em Python evitando loops em Python puro, por exemplo usando vetores do Numpy. A biblioteca Numpy é uma biblioteca pré-compilada e otimizada para operações de álgebra linear, portanto, quando usamos sua aritmética estamos usando recursos computacionais de forma eficiente.

Mais informações no tutorial: [link privado removido]

Vejamos 2 pequenas mudanças no código de eliminação Gaussiana:

In [ ]:
# Método Clássico de Eliminação Gaussiana com vetorização
#
# O método recebe como entrada uma matrix aumentada a=[A|b],
# com dimensões n linhas por n+1 colunas, tendo b na última
#
# O método retorna a solução do sistema linear Ax=b,
#            ou um erro quando isso não for possível
def elim_gauss_vet(a_orig):
  #Faço uma cópia, para não estragar a matriz original
  a = np.copy(a_orig)

  #Número de linhas e colunas
  n,m = a.shape

  # Vetor da solução
  x = np.empty((n))

  #Verifica se usuário forneceu uma matriz no formato certo
  if m != n+1 :
    print("Essa matriz não tem dimensões adequadas:", n, m)
    return x

  zero_eps = 10e-16
  # Escalonamento
  for j in range(n): #Colunas
      #Caso o pivot seja zero (ou próximo de zero)
      if np.abs(a[j,j]) < zero_eps:
          #Pivotar linha i com linha linha com pivot não nulo!
          ktroca = j
          for k in range(j+1,n):
              if np.abs(a[k,j]) > zero_eps:
                  ktroca = k
                  break
          #Troca as linhas
          if (ktroca != j):
            print("Troquei linha ", j, " pela linha ", ktroca)
            a[[j, ktroca], :] = a[[ktroca, j], :]
          else:
            print("Esse sistema tem determinante nulo, ou muito perto de zero.")
            return x

      #Escalonamento
      for i in range(j+1, n):
          ratio = a[i,j]/a[j,j]
          #Vetorizei aqui!
          #for k in range(n+1):
          #    a[i,k] = a[i,k] - ratio * a[j,k]
          a[i,:] -= ratio * a[j,:]

  # Substituição
  x[n-1] = a[n-1,n]/a[n-1,n-1]

  for i in range(n-2,-1,-1):
      #Vetorizei aqui!
      #x[i] = a[i,n]
      #for j in range(i+1,n):
      #    x[i] = x[i] - a[i,j]*x[j]
      #x[i] = x[i]/a[i,i]
      x[i] = (a[i,n] - np.dot(a[i,i+1:n], x[i+1:n]))/a[i,i]

  return x

Testando com a mesma matriz de antes:

In [ ]:
A, b, a = meu_sistema_linear(300, magic_ilong)

start_time = time.time()
x_vet = elim_gauss_vet(a)
tempo_vet = time.time() - start_time
print("Solução:", x_vet[:10]) #imprime só alguns elementos da solução
print("Resíduo max|b-Ax|: ", np.max(np.abs(b-A@x_vet)), " Tempo que levou: ", tempo_vet, " segundos")
print()
print("Aceleração em relação ao método sem vetorização:", tempo_orig/tempo_vet, " vezes mais rápido")
print("Diferença máxima na solução:", np.max(np.max(x-x_vet)))

Observações:



*   O método ficou muito mais rápido!
*   A solução não ficou idêntica à solução anterior, pois a ordem das contas mudou um pouco, e portanto temos pequenos erros de arredondamento se acumulando de formas diferentes.



### Dá para vetorizar mais?

Uma forma é mudar um pouco o algoritmo. Vejamos um algoritmo de decomposição LU (que vai servir de base para Elim. Gaussiana vetorial): Algoritmo de Crout (ou Doolittle, dependendo de onde colocamos a diagonal unitária).

O problema agora é resolver

$$ Ax=LUx=b $$


In [ ]:
# Método Clássico de Eliminação Gaussiana via decomposição LU
#
# O método recebe como entrada uma matrix A (n x n) e o vetor b (n x 1),
#
# O método retorna a solução do sistema linear Ax=b,
#            ou um erro quando isso não for possível
# Devolve tambem as matrizes L e U da decomposição A=LU
def elim_gauss_LU(A, b):

  #Número de linhas e colunas
  n,m = A.shape

  # Vetor da solução
  x = np.empty((n))

  #Verifica se usuário forneceu uma matriz no formato certo
  if m != n :
    print("Essa matriz não tem dimensões adequadas:", n, m)
    return x

  zero_eps = 10e-16

  #Guardo tanto L quanto U em uma única matriz!!!!
  LU = np.eye(n) #Matriz identidade

  # @ faz multiplicação de matrizes usando numpy
  for i in range(n):

      #Varre linhas superiores (Upper)
      LU[i,i:] = A[i,i:]-LU[i,:i] @ LU[:i,i:]

      #Varre colunas inferiores (Lower)
      if np.abs(LU[i,i]) <zero_eps:
        print('Ainda não implementei pivotamento :-( ')
        return x
      LU[(i+1):,i] = ( A[(i+1):,i]- LU[(i+1):,:i] @ LU[:i,i] ) / LU[i,i]

  # Substituição
  # LUx=b =>  Ly=b,  Ux=y
  y = np.zeros(n)
  # Ly=b
  y[0] = b[0]
  for i in range(1,n,1):
      #Vetorizei aqui!
      y[i] = (b[i] - np.dot(LU[i,:i], y[:i]))

  # Ux=y
  x = np.zeros(n)
  x[n-1] = y[n-1]/LU[n-1,n-1]
  for i in range(n-2,-1,-1):
      #Vetorizei aqui!
      x[i] = (y[i] - np.dot(LU[i,i+1:], x[i+1:]))/LU[i,i]

  #Forma matrizes L e U, tais que A=LU
  U = np.triu(LU) # Pega só triangular superior
  L = np.tril(LU) # Pega só triangular inferior
  np.fill_diagonal(L, 1.0) # Preenche com 1 a diagonal inferior
  return x, L, U

Testando:

In [ ]:
A, b, a = meu_sistema_linear(300, magic_ilong)
start_time = time.time()
x_LU, L, U = elim_gauss_LU(A, b)
tempo_LU = time.time() - start_time
print("Solução:", x_LU[:10]) #imprime só alguns elementos da solução
print("Resíduo max|b-Ax|: ", np.max(np.abs(b-A@x_LU)), " Tempo que levou: ", tempo_LU, " segundos")
print()
print("Aceleração em relação ao método sem vetorização:", tempo_orig/tempo_LU, " vezes mais rápido")
print("Diferença máxima na solução:", np.max(np.max(x-x_LU)))
print('\nTeste LU: ')
print(np.max(np.max(np.abs(L@U-A))))

Impressionante a diferença de performance, não? ;-)







---



#### 🧩*Desafio Opcional*

Você consegue usar essa ideia do LU para fazer diretamente a solução do sistema linear vetorial, com apenas um loop, sem precisar calcular L e U explicitamente como fizemos?



---



## Precisão reduzida

Gostaríamos de simular o caso de termos uma precisão (número de algarismos significativos) diferentes.

A seguir temos uma implementação análoga à apresentada no primeiro código de eliminação de Gauss (a função elim_gauss()), mas que agora arredonda cada operação feita para um certo número de algarismos significations. Aqui usamos a função round(número, sigfigs) após cada conta que realizar no algoritmo. TODAS as contas de +-*/ devem ser sempre arredondadas!


In [ ]:
# Essa função já foi declarada anteriormente!

#from math import floor, log10
# Uma opção para simular algarismos significativos
#def round_alg_sig(x, sig_figs):
#    # Recebe
#    #    - x: numero do tipo float
#    #    - sig_figs: número de algarismos significativos a ser usado
#    # Devolve
#    #    - Número arredondado para a aprecisão desejada#
#
#    if abs(x) > 0.0 and np.isfinite(x):
#      x = float(x)
#      precision = int(sig_figs)
#      # Uso o round de casas decimais, deslocando as casas decimais para a posição correta
#      return round(x, -int(floor(log10(abs(x)))) + (precision - 1))
#    else:
#      return x

# Outra opção - automatizada
#!pip install sigfig
#from sigfig import round
#round(3.14159, sigfigs=2)

In [ ]:
# Método Clássico de Eliminação Gaussiana com aritmética de ponto flutuante restrita
#
# O método recebe como entrada uma matrix aumentada a=[A,b] referente ao
#    sistema linear Ax=b, com A (n x n) e o vetor b (n x 1),
# O método recebe também o número de algarismos significativos a serem usados nas contas
#
# O método retorna a solução do sistema linear Ax=b,
#            ou um erro quando isso não for possível
def elim_gauss_precisao_reduzida(a_orig, sigfig=17):

  #Faço uma cópia, para não estragar a matriz original
  a = np.copy(a_orig)

  #Número de linhas e colunas
  n,m = a.shape

  # Vetor da solução
  x = np.empty((n))

  #Verifica se usuário forneceu uma matriz no formato certo
  if m != n+1 :
    print("Essa matriz não tem dimensões adequadas:", n, m)
    return x

  zero_eps = 10e-16
  #Escalonamento
  for j in range(n): #colunas

    #Caso o pivot seja zero (ou próximo de zero)
    if np.abs(a[j,j]) < zero_eps:
      #Pivotar linha i com linha linha com pivot não nulo!
      ktroca = j
      for k in range(j+1,n):
          if np.abs(a[k,j]) > zero_eps:
              ktroca = k
              break
      #Troca as linhas
      if (ktroca != j):
        print("Troquei linha ", j, " pela linha ", ktroca)
        a[[j, ktroca], :] = a[[ktroca, j], :]
      else:
        print("Esse sistema tem determinante nulo, ou muito perto de zero.")
        return x


    for i in range(j+1, n): #linhas
        ratio = round_alg_sig(a[i, j]/a[j, j],  sigfig)
        for k in range(n+1):
            a[i,k] = round_alg_sig(a[i,k] - round_alg_sig(ratio * a[j,k], sigfig), sigfig)


  # Substituição
  x[n-1] = round_alg_sig(a[n-1,n]/a[n-1,n-1], sigfig)
  for i in range(n-2,-1,-1):
      x[i] = a[i,n]
      for j in range(i+1,n):
          x[i] = round_alg_sig(x[i] - round_alg_sig(a[i,j]*x[j], sigfig), sigfig)
      x[i] = round_alg_sig(x[i]/a[i,i], sigfig)

  return x


###❗Atividade 2 ❗

Teste a resolução do seu sistema linear personalizado com dimensão 30 variando o número de algarísmos significativos e reporte os seguintes casos:

1. Erro com 7 algarismos significativos
2. Erro com 8 algarismos significativos
3. Erro com 9 algarismos significativos

Quantos algarismos significativos precisamos para ficarmos com um erro menor que 10e-4?

In [ ]:
import numpy as np
from math import floor, log10

# Função para arredondar para um número específico de algarismos significativos
def round_alg_sig(x, sig_figs):
    if abs(x) > 0.0 and np.isfinite(x):
        x = float(x)
        precision = int(sig_figs)
        return round(x, -int(floor(log10(abs(x)))) + (precision - 1))
    else:
        return x

# Método de eliminação gaussiana com precisão reduzida
def elim_gauss_precisao_reduzida(a_orig, sigfig=17):
    a = np.copy(a_orig)
    n, m = a.shape
    x = np.empty((n))
    if m != n + 1:
        print("Essa matriz não tem dimensões adequadas:", n, m)
        return x
    zero_eps = 10e-16
    for j in range(n):
        if np.abs(a[j, j]) < zero_eps:
            ktroca = j
            for k in range(j + 1, n):
                if np.abs(a[k, j]) > zero_eps:
                    ktroca = k
                    break
            if ktroca != j:
                print("Troquei linha ", j, " pela linha ", ktroca)
                a[[j, ktroca], :] = a[[ktroca, j], :]
            else:
                print("Esse sistema tem determinante nulo, ou muito perto de zero.")
                return x
        for i in range(j + 1, n):
            ratio = round_alg_sig(a[i, j] / a[j, j], sigfig)
            for k in range(n + 1):
                a[i, k] = round_alg_sig(a[i, k] - round_alg_sig(ratio * a[j, k], sigfig), sigfig)
    x[n - 1] = round_alg_sig(a[n - 1, n] / a[n - 1, n - 1], sigfig)
    for i in range(n - 2, -1, -1):
        x[i] = a[i, n]
        for j in range(i + 1, n):
            x[i] = round_alg_sig(x[i] - round_alg_sig(a[i, j] * x[j], sigfig), sigfig)
        x[i] = round_alg_sig(x[i] / a[i, i], sigfig)
    return x


dim = 30
np.random.seed(42)
A = np.random.rand(dim, dim) * 10 # Cria uma matriz A (30×30) com valores aleatórios entre 0 e 10
b = np.random.rand(dim) * 10 # Cria um vetor b (30×1) com valores aleatórios entre 0 e 10
Ab = np.hstack((A, b.reshape(-1, 1))) # Conecta A e b e cria uma matriz aumentada

# Teste da resolução com diferentes precisões
for sigfig in [7, 8, 9]:
    x_aprox = elim_gauss_precisao_reduzida(Ab, sigfig)
    erro = np.linalg.norm(A @ x_aprox - b)
    print(f"Erro com {sigfig} algarismos significativos: {erro:.2e}")

# erro < 10^-4
sigfig_necessario = 7
while True:
    x_aprox = elim_gauss_precisao_reduzida(Ab, sigfig_necessario)
    erro = np.linalg.norm(A @ x_aprox - b) # O termo A @ x_aprox calcula o lado esquerdo do sistema Ax
    if erro < 1e-4:
        break
    sigfig_necessario += 1

print(f"Precisamos de pelo menos {sigfig_necessario} algarismos significativos para obter erro < 10^-4.")

Resposta:

Erro com 7 algarismos significativos: 2.27e-02
Erro com 8 algarismos significativos: 3.62e-03
Erro com 9 algarismos significativos: 1.52e-04

Precisamos de pelo menos 10 algarismos significativos para obter erro < 10^-4.

Podemos concluir que o erro numérico diminui conforme aumentamos o número de algarismos significativos na aritmética de ponto flutuante. Assim, o método de eliminação gaussiana é afetado pela precisão numérica, tornando os resultados menos confiáveis quando os cálculos são truncados para um número menor de dígitos.



---



## Pivotamento Parcial
Reduzindo erros de artitmética de ponto flutuante permutando linhas.

A *condensação pivotal parcial* consiste em escolher como pivô, a cada passo do escalonamento, o elemento na coluna-pivô (da diagonal principal para baixo) com maior valor absoluto, de modo que todos os multiplicadores tenham módulo menor ou igual a 1.

O algoritmo consiste na seguinte estratégia:
1. Na eliminação de Gauss o pivô é termo da diagonal que será usado como divisor para chegarmos no multiplicador. Queremos que o pivô usado para construção do multiplicador seja o maior possível, para que o multiplicador seja menor que 1.
2. A cada nova coluna que vou escalonar (digamos que seja a coluna j, com o pivô sendo $a_{jj}$), verifico o valor em módulo dos elementos abaixo do pivô (elementos $a_{kj}$, $k\geq j$). Escolho a linha ($k$) que tem o elemento com maior valor em módulo, e troco com a linha principal ($j$), isto é, troco a linha j com a k.  
3. Prossigo com a estratégia usual de escalonamento com a matriz com as linhas trocadas. Note que solução do sistema não é alterada pela troca de linhas. Além disso, note ainda que não "estragamos" o escalonamento anterior, pois sempre trocamos com linhas abaixo do pivô.

Para mais detalhes, veja a video-aula [link privado removido]



---





Verifique que com sigfig=17 e sem pivotamento o resultado obtido com esse método é identico ao da nossa implementação ingênua.



### ❗Atividade 3❗



1. Implemente uma modificação do algoritmo de eliminação de Gauss com precisão reduzida agora considerando um **pivotamento parcial**, trocando as linhas nas etapas do escalonamento de forma a sempre obter um multiplicador em módulo menor que 1.

2. Verifique que com sigfig=17 e sem pivotamento o resultado obtido com esse método é identico ao da nossa implementação ingênua.

3. Resolva o seu problema personalizado com n=30 com 8, 9, 10 algarismos significativos. Compare os erros e resíduos obtidos com e sem pivotamento.

In [ ]:
#Código - Pode usar como referência o código anterior "elim_gauss_precisao_reduzida"

import numpy as np
from math import floor, log10

# Função para arredondar para um número específico de algarismos significativos
def round_alg_sig(x, sig_figs):
    if abs(x) > 0.0 and np.isfinite(x):
        x = float(x)
        precision = int(sig_figs)
        return round(x, -int(floor(log10(abs(x)))) + (precision - 1))
    else:
        return x

# Método de eliminação gaussiana com pivotamento parcial e precisão reduzida
def elim_gauss_pivotamento(a_orig, sigfig=17):
    a = np.copy(a_orig)
    n, m = a.shape
    x = np.empty((n))
    if m != n + 1:
        print("Essa matriz não tem dimensões adequadas:", n, m)
        return x
    zero_eps = 10e-16
    for j in range(n):
        max_row = j + np.argmax(np.abs(a[j:, j]))
        if j != max_row:
            print(f"Troquei linha {j} pela linha {max_row}")
            a[[j, max_row], :] = a[[max_row, j], :]
        if np.abs(a[j, j]) < zero_eps:
            print("Esse sistema tem determinante nulo, ou muito perto de zero.")
            return x
        for i in range(j + 1, n):
            ratio = round_alg_sig(a[i, j] / a[j, j], sigfig)
            for k in range(n + 1):
                a[i, k] = round_alg_sig(a[i, k] - round_alg_sig(ratio * a[j, k], sigfig), sigfig)
    x[n - 1] = round_alg_sig(a[n - 1, n] / a[n - 1, n - 1], sigfig)
    for i in range(n - 2, -1, -1):
        x[i] = a[i, n]
        for j in range(i + 1, n):
            x[i] = round_alg_sig(x[i] - round_alg_sig(a[i, j] * x[j], sigfig), sigfig)
        x[i] = round_alg_sig(x[i] / a[i, i], sigfig)
    return x

# Teste da resolução com diferentes precisões, comparando com e sem pivotamento
dim = 30
np.random.seed(42)
A = np.random.rand(dim, dim) * 10
b = np.random.rand(dim) * 10
Ab = np.hstack((A, b.reshape(-1, 1)))

print("--- Sem pivotamento ---")
for sigfig in [8, 9, 10]:
    x_aprox = elim_gauss_precisao_reduzida(Ab, sigfig)
    erro = np.linalg.norm(A @ x_aprox - b)
    print(f"Erro com {sigfig} algarismos significativos: {erro:.2e}")

print("\n--- Com pivotamento ---")
for sigfig in [8, 9, 10]:
    x_aprox_pivot = elim_gauss_pivotamento(Ab, sigfig)
    erro_pivot = np.linalg.norm(A @ x_aprox_pivot - b)
    print(f"Erro com {sigfig} algarismos significativos e pivotamento: {erro_pivot:.2e}")
    print()

# erro < 10^-4 com pivotamento
sigfig_necessario_pivot = 8
while True:
    x_aprox_pivot = elim_gauss_pivotamento(Ab, sigfig_necessario_pivot)
    erro_pivot = np.linalg.norm(A @ x_aprox_pivot - b)
    if erro_pivot < 1e-4:
        break
    sigfig_necessario_pivot += 1

print(f"Precisamos de pelo menos {sigfig_necessario_pivot} algarismos significativos para obter erro < 10^-4 com pivotamento.")


Coloque aqui observações sobre o experimento realizado:

* A precisão da solução melhorou com o pivotamento? Quanto?
* Isso depende da precisão das contas?
* 🧩 Será que para qualquer matriz o erro vai sempre melhorar?
* 🧩 Consegue ilustrar algum teste que o pivotamento não ajuda?


O experimento demonstrou que o uso do pivotamento parcial na eliminação gaussiana melhora bastante a precisão da solução, especialmente quando a precisão numérica é reduzida. Sem o pivotamento, o erro pode crescer rapidamente devido à amplificação de pequenos erros ao longo do escalonamento. O pivotamento evita que divisões sejam feitas por números muito pequenos, reduzindo a propagação de erros numéricos.

Por outro lado, a eficácia do pivotamento depende da estrutura da matriz. Em matrizes bem condicionadas, onde os coeficientes já possuem uma distribuição equilibrada, o impacto do pivotamento pode ser insignificante. Além disso, existem casos onde o pivotamento não traz benefícios expressivos, como em sistemas já bem escalonados. Para ilustrar um caso onde o pivotamento não é tão eficiente, podemos considerar matrizes diagonalmente dominantes, nas quais os elementos da diagonal já são os maiores da respectiva coluna. Nesses casos, a troca de linhas não altera significativamente o comportamento do sistema, resultando em erros semelhantes com e sem pivotamento.



---

